In [1]:
import pandas as pd
import joblib  # Used for saving feature name list (install via: pip install joblib)

# ----------------------------------------------------------------------------------
# 1. Initial Setup and Data Loading
# ----------------------------------------------------------------------------------
file_path = '202207_corpor_CB.csv'
df = pd.read_csv(file_path)

# Define independent variable (X) list for analysis (64 variables total)
feature_cols = [
    'FN1_1', 'FN1_2', 'FN1_3', 'FN1_4', 'FN1_5', 'FN1_6', 'FN1_7', 'FN1_8',
    'FN1_9', 'FN1_10', 'FN1_11', 'FN1_11_1', 'FN1_11_2', 'FN1_11_3', 'FN1_11_4',
    'FN1_13', 'FN1_13_1', 'FN1_14', 'FN1_15', 'FN1_16', 'FN1_17', 'FN1_18',
    'FN1_19', 'FN1_20', 'FN1_21', 'FN1_21_1', 'FN1_22', 'FN1_22_1', 'FN1_22_2',
    'FN1_23', 'FN1_24', 'FN1_24_1', 'FN2_1', 'FN2_1_1', 'FN2_2', 'FN2_2_1',
    'FN2_3', 'FN2_3_1', 'FN2_3_2', 'FN2_3_3', 'FN2_3_4', 'FN2_3_5', 'FN2_4',
    'FN2_5', 'FN2_5_1', 'FN2_7', 'FN2_8', 'FN2_9', 'FN2_10', 'FN2_10_1',
    'FN3_1', 'FN3_2', 'FN3_2_1', 'FN3_2_2', 'FN3_3', 'FN3_4_1', 'FN3_4_2',
    'FN3_6', 'FN3_7', 'FN3_8', 'FN3_10', 'FN3_10_1', 'FN3_11', 'FN3_11_1'
]

# Identifier and target variable
id_col = 'ID'
target_col = 'PERF_12M'

# Full column list required
selected_cols = [id_col] + feature_cols + [target_col]

print("Data loaded successfully:", df.shape)

# ----------------------------------------------------------------------------------
# 2. EDA: Target Industry Selection (by bankruptcy count)
# ----------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("EDA: Bankruptcy Analysis by (Audit Classification + Industry) Combination")
print("=" * 80)

# Grouping and aggregation
combination_default = df.groupby(['WG_GB', 'SIC_CD_3']).agg({
    target_col: ['sum', 'count', 'mean']
})
combination_default.columns = ['Bankruptcy_Count', 'Total_Count', 'Bankruptcy_Rate']
combination_default = combination_default.sort_values('Bankruptcy_Count', ascending=False)

# Automatically select top combination
top_combination = combination_default.iloc[0]
top_wg, top_industry = combination_default.index[0]

print(f"Top combination selected:")
print(f"- Audit Classification: {top_wg}")
print(f"- Industry Code: {top_industry} (Wholesale and commission trade, etc.)")
print(f"- Bankruptcy Count: {int(top_combination['Bankruptcy_Count'])}")
print(f"- Total Count: {int(top_combination['Total_Count'])}")
print(f"- Bankruptcy Rate: {top_combination['Bankruptcy_Rate']:.2%}")

# ----------------------------------------------------------------------------------
# 3. Data Filtering (Retain full data — no undersampling here)
# ----------------------------------------------------------------------------------
# Extract only records matching the selected combination
final_df = df[(df['WG_GB'] == top_wg) & (df['SIC_CD_3'] == top_industry)][selected_cols].copy()

# Check for missing values
if final_df.isnull().sum().sum() == 0:
    print("\nMissing value check: None (Clean Data)")
else:
    print("\n[Warning] Missing values detected. Preprocessing may be required.")
    print(final_df.isnull().sum()[final_df.isnull().sum() > 0])

# ----------------------------------------------------------------------------------
# 4. File Export (CSV + PKL)
# ----------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("File Export Process")
print("=" * 80)

# (1) Save modeling dataset (CSV)
csv_filename = 'selected_data_for_modeling.csv'
final_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print(f"1. Dataset saved: {csv_filename}")
print(f"   - Shape: {final_df.shape}")
print(f"   - Bankruptcy cases included: {final_df[target_col].sum()}")

# (2) Save feature name list (PKL) — Important!
# Prevents column ordering or omission issues in downstream steps (Step 2, 3)
pkl_filename = 'feature_names.pkl'
joblib.dump(feature_cols, pkl_filename)
print(f"2. Feature name list saved: {pkl_filename}")
print(f"   - Number of features saved: {len(feature_cols)}")

print("\n[Step 1 Complete] Proceed to the next step (Step2_Modeling).")

Data loaded successfully: (150000, 109)

EDA: Bankruptcy Analysis by (Audit Classification + Industry) Combination
Top combination selected:
- Audit Classification: 2
- Industry Code: G46 (Wholesale and commission trade, etc.)
- Bankruptcy Count: 266
- Total Count: 15482
- Bankruptcy Rate: 1.72%

Missing value check: None (Clean Data)

File Export Process
1. Dataset saved: selected_data_for_modeling.csv
   - Shape: (15482, 66)
   - Bankruptcy cases included: 266
2. Feature name list saved: feature_names.pkl
   - Number of features saved: 64

[Step 1 Complete] Proceed to the next step (Step2_Modeling).
